# Fuel Blending Problem with AMPL in Python
## AMPL Modeling for Book Problem 3.5

This notebook models the fuel-blending problem from book section `3.5` with AMPL from Python using `amplpy`.


## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

If your environment does not have `amplpy`, install it first:

```python
%pip install amplpy
```

The first call to `ampl_notebook(modules=["highs"], license_uuid="default")` may download the solver module if it is not already available.


In [1]:
from __future__ import annotations

from functools import wraps
from math import isclose
from time import perf_counter


In [2]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed

    return wrapper


def create_ampl_instance(solver: str = "highs"):
    ampl = ampl_notebook(modules=[solver], license_uuid="default")
    ampl.option["solver"] = solver
    ampl.option["solver_msg"] = 0
    return ampl


def round_if_close(value: float, digits: int = 4):
    rounded = round(value, digits)
    return int(round(rounded)) if isclose(rounded, round(rounded), abs_tol=1e-9) else rounded


def extract_2d_solution(variable, row_keys, col_keys):
    raw = variable.get_values().to_dict()
    values = {}
    for row in row_keys:
        for col in col_keys:
            values[(row, col)] = round_if_close(float(raw[(row, col)]))
    return values


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Problem: Blending Three Crudes into Three Gasolines

The company must satisfy the daily demand for `super` and `normal` gasoline while using the remaining budget to maximize the production of `euro`. The blending constraints are written exactly as composition bounds on the final gasoline mixtures.


In [3]:
GASOLINES = ["super", "normal", "euro"]
CRUDES = ["A", "B", "C"]
COMPONENTS = [1, 2, 3]

CRUDE_COMPOSITION = {
    ("A", 1): 0.80, ("A", 2): 0.10, ("A", 3): 0.05,
    ("B", 1): 0.45, ("B", 2): 0.30, ("B", 3): 0.20,
    ("C", 1): 0.30, ("C", 2): 0.40, ("C", 3): 0.25,
}
GASOLINE_SPEC = {
    ("super", 1): 0.60, ("super", 2): 0.25, ("super", 3): 0.10,
    ("normal", 1): 0.30, ("normal", 2): 0.20, ("normal", 3): 0.15,
    ("euro", 1): 0.40, ("euro", 2): 0.35, ("euro", 3): 0.20,
}
COST = {"A": 250, "B": 120, "C": 200}
AVAILABILITY = {"B": 300, "C": 700}
DEMAND = {"super": 60, "normal": 70}
MIN_A = 10
BUDGET = 50_000
EXPECTED = {
    ("super", "A"): 25.7143,
    ("super", "B"): 34.2857,
    ("super", "C"): 0.0,
    ("normal", "A"): 35.0,
    ("normal", "B"): 35.0,
    ("normal", "C"): 0.0,
    ("euro", "A"): 0.0,
    ("euro", "B"): 82.8348,
    ("euro", "C"): 82.8348,
}
EXPECTED_OBJECTIVE = 165.6696


@timed
def solve_fuel_blending_with_ampl(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        set GASOLINES ordered;
        set CRUDES ordered;
        set COMPONENTS ordered;

        param crude_composition {CRUDES, COMPONENTS} >= 0;
        param gasoline_spec {GASOLINES, COMPONENTS} >= 0;
        param crude_cost {CRUDES} >= 0;
        param availability {c in CRUDES: c != "A"} >= 0;
        param demand {g in GASOLINES: g != "euro"} >= 0;
        param min_a >= 0;
        param budget >= 0;

        var Blend {g in GASOLINES, c in CRUDES} >= 0;

        maximize EuroProduction:
            sum {c in CRUDES} Blend["euro", c];

        subject to SuperLow {k in {1, 3}}:
            sum {c in CRUDES} crude_composition[c, k] * Blend["super", c]
            >= gasoline_spec["super", k] * sum {c in CRUDES} Blend["super", c];

        subject to SuperHigh:
            sum {c in CRUDES} crude_composition[c, 2] * Blend["super", c]
            <= gasoline_spec["super", 2] * sum {c in CRUDES} Blend["super", c];

        subject to NormalHigh {k in {2, 3}}:
            sum {c in CRUDES} crude_composition[c, k] * Blend["normal", c]
            <= gasoline_spec["normal", k] * sum {c in CRUDES} Blend["normal", c];

        subject to NormalLow:
            sum {c in CRUDES} crude_composition[c, 1] * Blend["normal", c]
            >= gasoline_spec["normal", 1] * sum {c in CRUDES} Blend["normal", c];

        subject to EuroLow {k in {2, 3}}:
            sum {c in CRUDES} crude_composition[c, k] * Blend["euro", c]
            >= gasoline_spec["euro", k] * sum {c in CRUDES} Blend["euro", c];

        subject to EuroHigh:
            sum {c in CRUDES} crude_composition[c, 1] * Blend["euro", c]
            <= gasoline_spec["euro", 1] * sum {c in CRUDES} Blend["euro", c];

        subject to AvailabilityLimit {c in CRUDES: c != "A"}:
            sum {g in GASOLINES} Blend[g, c] <= availability[c];

        subject to BudgetLimit:
            sum {g in GASOLINES, c in CRUDES} crude_cost[c] * Blend[g, c] <= budget;

        subject to MinPurchaseA:
            sum {g in GASOLINES} Blend[g, "A"] >= min_a;

        subject to DemandSatisfaction {g in GASOLINES: g != "euro"}:
            sum {c in CRUDES} Blend[g, c] >= demand[g];
        '''
    )
    ampl.set["GASOLINES"] = GASOLINES
    ampl.set["CRUDES"] = CRUDES
    ampl.set["COMPONENTS"] = COMPONENTS
    ampl.param["crude_composition"] = CRUDE_COMPOSITION
    ampl.param["gasoline_spec"] = GASOLINE_SPEC
    ampl.param["crude_cost"] = COST
    ampl.param["availability"] = AVAILABILITY
    ampl.param["demand"] = DEMAND
    ampl.param["min_a"] = MIN_A
    ampl.param["budget"] = BUDGET
    ampl.solve()

    blend = extract_2d_solution(ampl.var["Blend"], GASOLINES, CRUDES)
    return {"blend": blend, "objective": round_if_close(ampl.obj["EuroProduction"].value())}


ampl_result, ampl_time = solve_fuel_blending_with_ampl()
print("=== FUEL BLENDING RESULTS WITH AMPL ===")
print(f"Solution -> {ampl_result}")
print(f"Time     -> {ampl_time:.8f} seconds")

for key, expected_value in EXPECTED.items():
    assert abs(ampl_result["blend"][key] - expected_value) <= 1e-3
assert abs(ampl_result["objective"] - EXPECTED_OBJECTIVE) <= 1e-3
print("AMPL matches the published blending plan within a 1e-3 tolerance.")


/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: === FUEL BLENDING RESULTS WITH AMPL ===
Solution -> {'blend': {('super', 'A'): 25.7143, ('super', 'B'): 34.2857, ('super', 'C'): 0, ('normal', 'A'): 35, ('normal', 'B'): 35, ('normal', 'C'): 0, ('euro', 'A'): 0, ('euro', 'B'): 82.8348, ('euro', 'C'): 82.8348}, 'objective': 165.6696}
Time     -> 1.68828829 seconds
AMPL matches the published blending plan within a 1e-3 tolerance.


## Variant Note

Section `3.5` proposes a student variation with a `5 %` blending loss. This notebook closes with the printed no-loss base model only.
